In [ ]:
# 1. pick a song by name
# 2. get that song's feature vector
# 3. compute cosine similarity between it and every other song
# 4. sort and return the top 10 most similar songs

In [5]:
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.metrics.pairwise import cosine_similarity


In [6]:
df = pd.read_csv("../data/processed/clean_tracks.csv")

In [7]:
features = ["danceability", "energy", "loudness", "speechiness",
            "acousticness", "instrumentalness", "liveness",
            "valence", "tempo"]
X = StandardScaler().fit_transform(df[features])

In [20]:
def recommend_similar(song_name, n=10, mode="mixed"):
    """
    Content-based recommender using cosine similarity on audio features.

    mode options:
    - "mixed": pure audio similarity, any genre (default)
    - "same_genre": only recommend songs from the same genre
    - "boosted": prefer same genre, but still allow strong cross-genre matches
    """
    matches = df[df["track_name"].str.lower() == song_name.lower()]
    if matches.empty:
        print("Song not found. Try another name.")
        return None
    idx = matches.index[0]
    genre = df.loc[idx, "track_genre"]

    song_vector = X[idx].reshape(1, -1)
    similarities = cosine_similarity(song_vector, X).flatten()

    scores = similarities.copy()

    if mode == "same_genre":
        mask = (df["track_genre"] == genre).values
        scores = scores * mask

    elif mode == "boosted":
        mask = (df["track_genre"] == genre).values
        scores = scores + (mask * 0.05)

    sorted_indices = scores.argsort()[::-1]
    sorted_indices = [i for i in sorted_indices if i != idx][:n]

    results = df.iloc[sorted_indices][["track_name", "artists", "track_genre"]].copy()
    results["similarity"] = similarities[sorted_indices]
    return results

In [21]:
print("Mixed (original):")
print(recommend_similar("Pal", mode="mixed"))



Mixed (original):
                            track_name  \
52713                             身騎白馬   
580          能古島の片想い - Remastered 2018   
31776              Quando Deus Se Cala   
25006                      What A Time   
16911  What A Time (feat. Niall Horan)   
27035                           Vienna   
26978                           Salaam   
31714                         Três Ais   
10875                               心事   
52905                              錯的人   

                                                 artists track_genre  \
52713                                           LaLa Hsu    mandopop   
580                                          Yosui Inoue    acoustic   
31776                                     Voz da Verdade      gospel   
25006                         Julia Michaels;Niall Horan     electro   
16911                         Julia Michaels;Niall Horan       dance   
27035                                         Billy Joel        folk   
26978  Salim–Su

In [22]:
print("\nSame genre only:")
print(recommend_similar("Pal", mode="same_genre"))




Same genre only:
                                              track_name  \
60194                                     Visiting Hours   
60166                                           Ik Lamha   
60341                                      The Scientist   
60343                                      Everyday Life   
60200    Mere Humsafar (Original Score) [Female Version]   
60267                              Hasi - Female Version   
60247                                    drivers license   
60282                 Mudhal Nee Mudivum Nee Title Track   
60243  Mudhal Nee Mudivum Nee Title Track (From "Mudh...   
60275                                          Munbe Vaa   

                         artists track_genre  similarity  
60194                 Ed Sheeran         pop    0.964607  
60166            Azaan Sami Khan         pop    0.951275  
60341                   Coldplay         pop    0.946035  
60343                   Coldplay         pop    0.938156  
60200              Yashal 

In [23]:
print("\nBoosted (blended):")
print(recommend_similar("Pal", mode="boosted"))


Boosted (blended):
                            track_name                     artists  \
60194                   Visiting Hours                  Ed Sheeran   
60166                         Ik Lamha             Azaan Sami Khan   
60341                    The Scientist                    Coldplay   
52713                             身騎白馬                    LaLa Hsu   
580          能古島の片想い - Remastered 2018                 Yosui Inoue   
60343                    Everyday Life                    Coldplay   
31776              Quando Deus Se Cala              Voz da Verdade   
16911  What A Time (feat. Niall Horan)  Julia Michaels;Niall Horan   
25006                      What A Time  Julia Michaels;Niall Horan   
27035                           Vienna                  Billy Joel   

      track_genre  similarity  
60194         pop    0.964607  
60166         pop    0.951275  
60341         pop    0.946035  
52713    mandopop    0.990601  
580      acoustic    0.989895  
60343         pop